In [1]:
import sys
print(sys.executable)

C:\Users\shlok\projects\ddp-llm\.venv\Scripts\python.exe


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
)
print("loaded. device:", model.device, "dtype:", model.dtype)
print("VRAM used (GB):", torch.cuda.memory_allocated() / 1e9)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shlok\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded. device: cuda:0 dtype: torch.float16
VRAM used (GB): 3.087429632


In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "In one sentence, what is comparison-based search?"},
]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to("cuda")

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False)

print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Comparison-based search involves comparing elements to find the desired item or solution.


In [1]:
import json
from pathlib import Path

path = Path(r"C:\Users\shlok\projects\ddp-llm\data\seed.jsonl")
examples = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"loaded {len(examples)} examples")
for i, ex in enumerate(examples):
    assert "options" in ex and "utterance" in ex and "label" in ex, f"missing field at line {i}"
    assert ex["options"] == ["A", "B", "C", "D"], f"bad options at line {i}"
    assert ex["label"] == "*" or (isinstance(ex["label"], list) and all(x in ex["options"] for x in ex["label"])), f"bad label at line {i}"
print("all valid")

loaded 50 examples
all valid
